# Analyse du refroidisseur d'acide de sechage E7301

**Atelier Sulfurique PS III — Maroc Chimie, OCP Group**
Mounir Sanbouli — Stage OCP, Programme Bionic

---

Ce notebook justifie, etape par etape, les choix de conception du systeme de
surveillance. Il ne se contente pas de tracer des courbes : chaque section
repond a une question qui a determine une decision technique.

1. Que contiennent reellement les donnees ?
2. Quels capteurs peut-on croire ?
3. Pourquoi une approche statistique classique echoue-t-elle ici ?
4. Pourquoi le residu de puissance ne pouvait pas marcher non plus ?
5. Ce qui debloque tout : le coefficient d'echange global UA
6. Que detecte le systeme sur 14 mois ?
7. Le Judge merite-t-il qu'on lui fasse confiance ?

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (13, 4.2), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False,
    "axes.spines.right": False, "font.size": 10,
})

from src.config import DATA_DIR
from src.domain.knowledge import load_domain
from src.ingest.dcs_loader import ingest

domain = load_domain()
print(domain.briefing_equipment())

## 1. Que contiennent reellement les donnees ?

L'export DCS ne s'accompagne d'aucun dictionnaire de tags. L'interpretation a ete
reconstruite a partir de trois sources convergentes : la nomenclature ISA-5.1,
la plage de valeurs observee, et le comportement des signaux lors des arrets.

Chaque interpretation est tracee dans `src/domain/tags.yaml` avec son niveau de
confiance et sa justification — elle est donc contestable et corrigeable par le
tuteur OCP sans toucher au code.

In [ ]:
res = ingest(DATA_DIR / "raw" / "DATA.xlsx", domain)
print(res.summary())

# `Tag.confidence` PORTE LES BASES DE DETERMINATION, PAS UN NIVEAU.
# Le champ a garde son nom en changeant de sens : il contient desormais la
# liste des bases jointes par des virgules — `isa_5_1,process,data` — et non
# plus `confirmed` / `inferred` / `unknown`, valeurs qui n'existent plus.
BASES = {
    "isa_5_1": "nomenclature ISA-5.1", "process": "physique du procede",
    "data": "comportement des donnees", "stoichio": "coherence stoechiometrique",
    "climatology": "climatologie",
}
for t in domain.monitored_tags:
    bases = [BASES.get(b, b) for b in t.confidence.split(",") if b]
    print(f"\n{t.alias:14s} {t.label}")
    print(f"   sens etabli par : {', '.join(bases)}")
    print("   " + " ".join((t.rationale or "").split())[:150])

## 2. Quels capteurs peut-on croire ?

Question preliminaire a toute modelisation. Un modele entraine sur un capteur mort
apprend du bruit d'instrumentation et le presente ensuite comme un diagnostic.

In [ ]:
h = res.sensor_health
display(h[["alias", "role", "availability_pct", "n_quality_code",
           "n_frozen", "n_saturated", "n_out_of_range"]])

In [ ]:
# Les deux defaillances averees, tracees sur toute la periode.
raw = pd.read_excel(DATA_DIR / "raw" / "DATA.xlsx")
raw["TIME"] = pd.to_datetime(raw["TIME"])
raw = raw.set_index("TIME")

fig, ax = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

s1 = pd.to_numeric(raw["S_MC_SULF_TI5303-4X_B"], errors="coerce")
ax[0].plot(s1.index, s1.values, lw=.7, color="#f85149")
ax[0].axhline(327.67, ls="--", color="k", lw=.9)
ax[0].annotate("butee d'echelle 327.67 = 32767/100\n(depassement entier 16 bits)",
               xy=(pd.Timestamp("2024-09-15"), 327.67), xytext=(pd.Timestamp("2024-03-01"), 250),
               arrowprops=dict(arrowstyle="->", lw=.9), fontsize=9)
ax[0].set_title("TI5303-4X — sature depuis aout 2024 : 7 mois de donnees mortes")
ax[0].set_ylabel("valeur")

s2 = pd.to_numeric(raw["S_MC_SULF_PHI5306X-3_B"], errors="coerce")
ax[1].plot(s2.index, s2.values, lw=.7, color="#d29922")
ax[1].axhline(-14.407, ls="--", color="k", lw=.9)
ax[1].annotate("fige a -14.407 pendant ~1900 h",
               xy=(pd.Timestamp("2024-02-01"), -14.4), xytext=(pd.Timestamp("2024-05-01"), -5),
               arrowprops=dict(arrowstyle="->", lw=.9), fontsize=9)
ax[1].set_title("PHI5306X-3 — signal fige sur les trois premiers mois, puis 139 codes qualite")
ax[1].set_ylabel("valeur")
plt.tight_layout(); plt.show()

In [ ]:
# Gel simultane de sept tags en juin 2024 : la simultaneite exclut une panne
# de capteur individuelle et designe une interruption de l'acquisition.
q = res.quality
jun = q[(q.issue == "FROZEN") & (q.timestamp >= "2024-06-01") & (q.timestamp <= "2024-06-15")]
print("Tags figes simultanement du 3 au 10 juin 2024 :")
print(jun.groupby("alias")["timestamp"].agg(["count", "min", "max"]).to_string())

## 3. Pourquoi une approche statistique classique echoue-t-elle ?

C'est le point de bascule du projet. La temperature de sortie acide est
**regulee** : sa distribution est ecrasee autour de la consigne. Un z-score sur
ce signal ne detecte rien tant que la regulation tient — et quand elle lache,
la degradation est deja consommee.

In [ ]:
run = res.readings[res.readings.process_state == "RUNNING"]
q = run["T_ACID_OUT"].quantile([.01, .25, .5, .75, .99])
print("Temperature de sortie acide en marche etablie :")
for k, v in q.items():
    print(f"  P{int(k*100):02d} = {v:.2f} degC")
print(f"\nAmplitude P1-P99 : {q[0.99] - q[0.01]:.2f} degC sur 14 mois")

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].hist(run["T_ACID_OUT"].dropna(), bins=90, color="#58a6ff")
ax[0].axvline(66, color="k", ls="--", label="consigne 66 degC")
ax[0].set_title("Sortie acide — variable REGULEE (ecart-type ~0.6 degC)")
ax[0].set_xlabel("degC"); ax[0].legend()

ax[1].hist(run["T_ACID_IN"].dropna(), bins=90, color="#f85149")
ax[1].set_title("Entree acide — variable LIBRE (ecart-type ~3 degC)")
ax[1].set_xlabel("degC")
plt.tight_layout(); plt.show()

**Conclusion de cette section.** L'encrassement du faisceau ne se lit pas sur le
*resultat* : la temperature de sortie est maintenue par la regulation.

Une premiere approche en concluait qu'il fallait lire l'*effort* — le residu de
puissance evacuee. La section 4 montre pourquoi cette conclusion etait fausse.

## 4. Pourquoi le residu de puissance ne pouvait pas marcher

Modeliser la puissance attendue puis suivre le residu paraissait naturel.
**Cette approche est fausse, et l'erreur est algebrique.**

La puissance est calculee par definition : `Q = rho.cp.V.(T_in - T_out)`. Le
modele la regresse sur le debit, l'entree et leur produit. Comme `T_out` est
regulee autour de 66 degC, la cible est deja une combinaison lineaire de deux
regresseurs presents : **la regression retrouve sa propre definition.**

La cellule suivante le mesure. Le chiffre qui compte n'est pas le R2, c'est
l'ecart entre le R2 appris et celui d'une reconstruction sans aucun
apprentissage.

In [ ]:
from src.features.e7301_features import build_features, independence_report

feats, refs = build_features(res.readings, res.quality, domain)

effort = refs.effort
print("REFERENCE D'EFFORT DE REGULATION — l'approche refutee")
print(f"  R2 appris            : {effort.r2:.4f}")
print(f"  R2 SANS apprentissage : {effort.naive_r2:.4f}")
print(f"  apport reel du modele : {effort.r2 - effort.naive_r2:+.4f}")

# Le residu d'effort EST l'ecart de consigne, change de signe.
ind = independence_report(feats)
for nom in ("regulation_effort_z", "ua_residual_z", "t_in_residual_z"):
    e = ind[nom]
    print(f"\n{nom:22s} corr(ecart de consigne) = {e['corr_control_deviation']:+.2f}"
          f"   independant : {e['independent']}")

## 5. Ce qui debloque tout : le coefficient d'echange global UA

L'encrassement se lit sur le **coefficient d'echange global UA**, et sur rien
d'autre. Le calculer exige la temperature du fluide froid, absente de l'export.

Elle n'est pourtant pas une inconnue : le refroidisseur est refroidi a l'eau de
mer, a **Safi**, ou le courant des Canaries et l'upwelling cotier maintiennent
17,0 degC en fevrier-mars et 22,0 degC en septembre. C'est une donnee
climatologique documentee, et surtout **exterieure a l'atelier** — aucune boucle
de regulation ne la contraint.

    epsilon = (T_in - T_out) / (T_in - T_eau_de_mer)
    NTU     = -ln(1 - epsilon)
    UA      = C_acide . NTU

**UA est un UA apparent** : le debit d'eau de mer n'est pas instrumente, et
c'est lui que la regulation manipule. La grandeur mesure donc l'etat de la
surface d'echange MULTIPLIE par l'action de la boucle froide.

In [ ]:
m = feats[feats.process_state == "RUNNING"].resample("MS").mean(numeric_only=True)

fig, ax = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
ax[0].plot(m.index, m.ua_kw_per_k, "o-", label="UA observe", color="#3fb950")
ax[0].plot(m.index, m.ua_expected, "s--", label="UA attendu", color="grey")
ax[0].set_ylabel("kW/K"); ax[0].legend()
ax[0].set_title("Coefficient d'echange global — observe contre attendu")

ax[1].bar(m.index, m.ua_residual_trend_14d, width=22,
          color=["#f85149" if v < 0 else "#58a6ff" for v in m.ua_residual_trend_14d])
seuil = -float(domain.modes["FAISCEAU_BOUCHAGE"].signature["warning_sigma"])
ax[1].axhline(seuil, ls="--", color="#f85149", lw=1)
ax[1].set_ylabel("sigma")
ax[1].set_title(f"Residu de UA — tendance 14 jours (seuil WARNING {seuil} sigma)")

ax[2].bar(m.index, m.fouling_resistance, width=22, color="#d29922")
ax[2].axhline(0, color="k", lw=.8)
ax[2].set_ylabel("K/kW")
ax[2].set_title("Resistance d'encrassement  Rf = 1/UA - 1/UA_attendu")
plt.tight_layout(); plt.show()

print(f"UA de reference : {refs.conductance.ua_reference:.2f} kW/K  "
      f"R2 {refs.conductance.r2:.3f}  sigma {refs.conductance.residual_std:.2f} kW/K")

### Le signe du residu — le point le plus delicat

| Configuration | Interpretation |
|---|---|
| Deficit de **UA** persistant, au-dela de 3 sigma | La surface transmet moins bien a conditions comparables -> **encrassement** |
| Exces d'**effort de regulation** et sortie sous consigne | La boucle froide travaille au-dela du necessaire -> **regime de conduite, pas une degradation** |

Confondre les deux conduirait a programmer un nettoyage haute pression du
faisceau — donc un arret de ligne de plusieurs jours — alors que l'echangeur
fonctionne mieux que sa reference.

`test_sur_refroidissement_est_un_regime_de_conduite` verrouille ce
comportement, et `test_effort_de_regulation_seul_ne_declare_pas_un_encrassement`
interdit la rechute.

## 5. Que detecte le systeme sur 14 mois ?

In [ ]:
from src.pipeline import E7301Pipeline

pipe = E7301Pipeline(use_llm=False)
scores = pipe.detector.score_series(pipe.features)
thr = pipe.detector.stat.threshold_

print(f"Heures analysees      : {len(scores)}")
print(f"Seuil de decision     : {thr:.3f}")
print(f"Heures atypiques      : {(scores >= thr).sum()} ({(scores >= thr).mean():.1%})")

ep = pipe.episodes()
print(f"\nApres agregation      : {len(ep)} episodes")
print(f"Reduction du volume   : facteur {int((scores >= thr).sum() / len(ep))}")
display(ep.head(10))

**Pourquoi l'agregation est indispensable.** Un exploitant ne traite pas 530
points d'alarme, il traite une dizaine d'evenements par mois. Sans cette etape,
le systeme serait desactive en salle de controle quelles que soient ses
performances statistiques.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.6))
ax.plot(scores.index, scores.values, lw=.4, color="#8b949e")
ax.axhline(thr, ls="--", color="#f85149", label=f"seuil {thr:.3f}")
for _, e in ep.head(10).iterrows():
    ax.axvspan(e["start"], e["end"], alpha=.22, color="#d29922")
ax.set_title("Score d'anomalie — les 10 episodes majeurs en surbrillance")
ax.set_ylabel("score"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Analyse detaillee du pic le plus marque.
peak = ep.iloc[1]["peak_at"]
a = pipe.analyze_at(peak)

print(f"INSTANT     {a.detection.timestamp}   (etat {a.detection.process_state})")
print(f"SEVERITE    {a.decision.severity}   score {a.detection.anomaly_score:.3f}")
print(f"\nCONSTATATIONS")
for f in a.detection.findings:
    print(f"  [{f.severity:8s}] {f.code:24s} {f.amdec_mode or '-'}")
    print(f"             {' '.join(f.message.split())[:130]}")
print(f"\nDIAGNOSTIC\n  {' '.join(a.decision.diagnosis.split())}")
print(f"\nACTION ({a.decision.recommended_action.urgency})\n  "
      f"{' '.join(a.decision.recommended_action.description.split())}")
print(f"\nJUDGE  {a.verdict.global_score:.2f}/10  "
      f"{'VALIDE' if a.verdict.agreement else 'REJETE'}")
for c in a.verdict.checks:
    print(f"  {'OK ' if c.passed else 'NOK'} {c.id:24s} {c.score:5.1f}/10  {c.detail[:90]}")

In [ ]:
# Explicabilite : contribution de chaque grandeur au score, par occlusion exacte.
if a.detection.attributions:
    df = pd.DataFrame(a.detection.attributions)
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.barh(df.feature[::-1], df.contribution[::-1], color="#58a6ff")
    ax.set_xlabel("chute du score si la grandeur etait normale")
    ax.set_title("Contribution au score d'anomalie (occlusion exacte)")
    plt.tight_layout(); plt.show()
    display(df)

## 6. Le Judge merite-t-il qu'on lui fasse confiance ?

**Le taux d'accord ne prouve rien.** Le Judge et l'agent deterministe raisonnent
sur la meme base de faits : un accord de 100 % est attendu et sans valeur
demonstrative. C'etait exactement le piege de la version precedente du projet.

La seule mesure valide consiste a soumettre au Judge des decisions
**deliberement fausses** et a verifier qu'il les detecte.

In [ ]:
from src.governance.judge_eval import TRAP_CASES, JudgeEvaluator

print(f"{len(TRAP_CASES)} types de faute au catalogue :\n")
for t in TRAP_CASES:
    print(f"  {t.name:28s} -> {t.expected_issue}")
    print(f"    {' '.join(t.description.split())[:120]}\n")

In [ ]:
ev = JudgeEvaluator(pipe).run(n_cases=12)
print(ev.report())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))

ax[0].hist(ev.clean.score, bins=18, range=(0, 10), alpha=.85,
           color="#3fb950", label="decisions saines")
ax[0].axvline(6.0, ls="--", color="k", label="seuil de validation")
ax[0].set_xlim(0, 10); ax[0].set_xlabel("note du Judge"); ax[0].legend()
ax[0].set_title("Le Judge ne rejette pas les decisions correctes")

t = ev.traps.sort_values("score_mean")
ax[1].barh(t.trap, t.score_mean, color="#f85149")
ax[1].axvline(6.0, ls="--", color="k")
ax[1].set_xlim(0, 10); ax[1].set_xlabel("note moyenne")
ax[1].set_title("Note attribuee a chaque type de faute injectee")
plt.tight_layout(); plt.show()

display(ev.traps)

### Ce que le banc a corrige dans le Judge

La premiere execution du banc a donne **65 % de detection** et revele trois
defauts precis, tous corriges :

1. **Codes d'anomalie ecrases** — un controle ne pouvait remonter qu'une seule
   anomalie ; quand une action etait a la fois sous-dimensionnee et dangereuse,
   la premiere disparaissait du journal d'audit.
2. **Tolerance de confiance trop laxiste** — une confiance de 0.99 sur des
   preuves justifiant 0.80 passait sous le radar (tolerance ramenee de 0.25 a 0.12).
3. **Etat de marche errone insuffisamment penalise** — detecte, mais avec un poids
   de 8 % la note restait a 9.08/10. Un plafond a 5.0 a ete introduit.

Detection portee de 65 % a 100 %. Cette boucle mesure-corrige-remesure est,
methodologiquement, le resultat le plus significatif du stage.

## 7. Ce que le systeme ne voit pas

Un systeme de surveillance qui ne declare pas ses angles morts donne une fausse
assurance. Ceux-ci sont exposes par l'API, affiches sur le dashboard, et le Judge
sanctionne tout diagnostic qui pretendrait les avoir detectes.

In [ ]:
print(domain.briefing_blind_spots())
print("\n" + "=" * 70)
print("Le mode de criticite la plus elevee de l'AMDEC (112) — degradation de")
print("l'anode sacrificielle — est structurellement invisible : aucune mesure de")
print("potentiel ou de courant de protection anodique ne figure dans l'export.")
print("Il reste couvert par le preventif : tache D (6 mois), tache E (3 ans).")